# Staging Area

## ORDERS

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE VIEW second_warehouse.staging.orders_view AS
SELECT order_id, customer_id, order_at AS order_date, region, total_usd AS order_cost
FROM second_warehouse.raw.raw_orders

In [ ]:
%%sql -r dataframe_2
SELECT * FROM second_warehouse.staging.orders_view

In [ ]:
%%sql -r dataframe_3
SELECT DISTINCT REGION FROM second_warehouse.staging.orders_view

In [ ]:
%%sql -r dataframe_4
UPDATE second_warehouse.raw.raw_orders SET REGION = 'EMEA' WHERE REGION = 'europe';

In [ ]:
%%sql -r dataframe_5
CREATE VIEW second_warehouse.staging.orders_clean_view AS
SELECT ROW_NUMBER() OVER(ORDER BY order_id ASC) AS S_No, *
FROM (SELECT DISTINCT *
FROM second_warehouse.staging.orders_view);

## Shipments

In [ ]:
CREATE OR REPLACE VIEW second_warehouse.staging.shipments_view AS
SELECT * FROM second_warehouse.raw.raw_shipments;

In [ ]:
SELECT * FROM second_warehouse.staging.shipments_view

In [ ]:
SELECT shipment_id, order_id, carrier,
    shipped_at::DATE AS shipping_date,
    delivered_at::DATE AS delivery_date,
    shipping_cost
FROM second_warehouse.staging.shipments_view;

In [ ]:
CREATE OR REPLACE VIEW second_warehouse.staging.shipments_clean_view AS (
SELECT *,
    CASE
        WHEN delivery_date IS NULL THEN 'Not Delivered'
        ELSE 'Delivered' END AS Delivery_status
FROM 
    (SELECT shipment_id, order_id, carrier,
    shipped_at::DATE AS shipping_date,
    delivered_at::DATE AS delivery_date,
    shipping_cost
FROM second_warehouse.staging.shipments_view));

# Mart

In [ ]:
CREATE OR REPLACE TABLE second_warehouse.marts.fct_fulfillment AS
SELECT *, order_date::DATE AS date_of_order
FROM second_warehouse.staging.orders_clean_view AS o
LEFT JOIN second_warehouse.staging.shipments_clean_view As s
USING(order_id);

In [ ]:
ALTER TABLE second_warehouse.marts.fct_fulfillment DROP COLUMN order_date;
ALTER TABLE second_warehouse.marts.fct_fulfillment RENAME COLUMN date_of_order TO order_date;

In [ ]:
CREATE OR REPLACE TABLE second_warehouse.marts.fct_lead_time AS
SELECT order_id, customer_id, region, carrier, delivery_date,
    DATEDIFF('day', order_date, shipping_date) AS lead_time_shipping, 
    DATEDIFF('day', shipping_date, delivery_date) AS lead_time_delivery
FROM second_warehouse.marts.fct_fulfillment;

In [ ]:
CREATE OR REPLACE TABLE second_warehouse.marts.fct_categorise_orders AS
SELECT order_id, customer_id, region, carrier, delivery_date, lead_time_delivery, lead_time_shipping,
    CASE
        WHEN delivery_date IS NULL AND lead_time_shipping >= 2 THEN 'LOST_OR_DELAYED'
        WHEN lead_time_delivery <= 1 THEN 'FAST_TRACK'
        ELSE 'STANDARD' END AS order_status
FROM second_warehouse.marts.fct_lead_time;

In [ ]:
CREATE OR REPLACE TABLE second_warehouse.marts.net_margin AS
SELECT order_id, region, order_cost, shipping_cost, (order_cost-shipping_cost) AS net_margin_USD
FROM second_warehouse.marts.fct_fulfillment;